In [18]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import torch
import torch.nn.functional as F
from PIL import Image

import json
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import classification_report, confusion_matrix

In [19]:
class DeepfakeJSONDataset(Dataset):
    def __init__(self, json_file, dataset_name, split='train', transform=None, frame_skip=1):
        """
        json_file: Chemin vers ton fichier .json
        dataset_name: Le nom de ton dataset racine dans le JSON (ex: "nomdudataset")
        split: "train" ou "test"
        transform: Les transformations PyTorch à appliquer
        frame_skip: Ne garde qu'une image sur N (ex: 10 pour garder 1 image sur 10)
        """
        self.transform = transform
        self.samples = []
        self.frame_skip = frame_skip # Ajout du paramètre
        
        self.label_map = {"FSAll_Fake": 0, "FRAll_Fake": 1}
        
        with open(json_file, 'r') as f:
            data = json.load(f)
            
        dataset_data = data.get(dataset_name, {})
        
        for manip_type, manip_data in dataset_data.items():
            if manip_type in self.label_map:
                int_label = self.label_map[manip_type]
                split_data = manip_data.get(split, {})
                
                for video_name, video_info in split_data.items():
                    frames = video_info.get("frames", [])
                    
                    # --- LA MAGIE EST ICI ---
                    # Au lieu de prendre toutes les frames, on utilise le slicing de Python
                    # liste[début:fin:pas] -> frames[::10] prendra 1 frame sur 10
                    sampled_frames = frames[::self.frame_skip]
                    
                    for frame_path in sampled_frames:
                        self.samples.append((frame_path, int_label))
                        
        print(f"Dataset '{split}' chargé ! {len(self.samples)} images retenues (Skip: 1/{self.frame_skip}).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_path, label = self.samples[idx]
        try:
            image = Image.open(frame_path).convert('RGB')
        except Exception as e:
            print(f"Erreur de lecture sur {frame_path}: {e}")
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [20]:
class DeepfakeRouter(nn.Module):
    def __init__(self):
        super(DeepfakeRouter, self).__init__()
        
        # 1. Le Filtre SRM 5x5 (Spatial Rich Model)
        # kernel_size passe à 5, et padding passe à 2 pour conserver la taille de l'image
        self.srm_filter = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, padding=2, groups=3, bias=False)
        
        # Voici la matrice exacte du filtre SRM d'ordre 3
        # Elle est conçue mathématiquement pour détruire l'image et isoler le bruit haute fréquence
        srm_kernel = torch.tensor([
            [-1,  2,  -2,  2, -1],
            [ 2, -6,   8, -6,  2],
            [-2,  8, -12,  8, -2],
            [ 2, -6,   8, -6,  2],
            [-1,  2,  -2,  2, -1]
        ], dtype=torch.float32) / 12.0  # Normalisation
        
        # On l'applique aux 3 canaux de couleur (RGB)
        self.srm_filter.weight.data = srm_kernel.view(1, 1, 5, 5).repeat(3, 1, 1, 1)
        self.srm_filter.weight.requires_grad = False # On gèle les poids !

        # On charge un modèle pré-entraîné très léger
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # 3. La tête de classification (0 = Swap, 1 = Reenactment)
        # On remplace la dernière couche (qui classait 1000 objets) par 2 classes
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, 2) # 2 classes : Swap vs Reenactment

    def forward(self, x):
        # L'image passe d'abord par le filtre pour ne garder que le bruit
        residual_noise = self.srm_filter(x)
        
        # Ensuite, le petit réseau analyse ce bruit
        out = self.backbone(residual_noise)
        return out

# Initialisation
router_model = DeepfakeRouter()
print("Modèle prêt ! Il attend des images de bruit pour trouver les artefacts.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/antoine/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████████████████████████████████| 44.7M/44.7M [00:04<00:00, 9.76MB/s]

Modèle prêt ! Il attend des images de bruit pour trouver les artefacts.


In [21]:
# 1. Préparation des données (on redimensionne pour MobileNet : 224x224)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = DeepfakeJSONDataset(
    json_file="/home/antoine/df-benchmark/datasets_json/FSvFR_ff.json",
    dataset_name="FSvFR_ff",
    split="train",
    transform=transform,
    frame_skip=15)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)

# 2. Configuration de l'entraînement
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
router_model = router_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(router_model.backbone.parameters(), lr=0.001)

# 3. Pseudo-boucle d'entraînement (à exécuter sur quelques époques)
router_model.train()
for epoch in range(10): 
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = router_model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
    print(f"Époque {epoch+1} terminée. Loss: {loss.item():.4f}")

Dataset 'train' chargé ! 44459 images retenues (Skip: 1/15).
Époque 1 terminée. Loss: 0.1682
Époque 2 terminée. Loss: 0.0131
Époque 3 terminée. Loss: 0.3593
Époque 4 terminée. Loss: 0.2052
Époque 5 terminée. Loss: 0.0186
Époque 6 terminée. Loss: 0.0391
Époque 7 terminée. Loss: 0.0760
Époque 8 terminée. Loss: 0.2683
Époque 9 terminée. Loss: 0.0007
Époque 10 terminée. Loss: 0.0112


In [22]:
router_model.eval()

test_data = DeepfakeJSONDataset(
    json_file="/home/antoine/df-benchmark/datasets_json/FSvFR_cdf.json",
    dataset_name="FSvFR_cdf",
    split="test",
    transform=transform,
    frame_skip=5)

test_loader = DataLoader(test_data, batch_size=32, shuffle=True, num_workers=4)

Dataset 'test' chargé ! 88853 images retenues (Skip: 1/5).


In [23]:
def evaluer_le_routeur(model, dataloader, device):
    print("Début de l'évaluation sur le set de validation...")
    
    # 1. RÈGLE D'OR N°1 : Mettre le modèle en mode évaluation
    # Cela désactive le Dropout et fige les couches de Batch Normalization
    model.eval() 
    
    correct_predictions = 0
    total_images = 0
    
    # Pour calculer des statistiques avancées (Precision, Recall)
    toutes_les_predictions = []
    tous_les_vrais_labels = []

    # 2. RÈGLE D'OR N°2 : Désactiver le calcul des gradients
    # Puisqu'on n'entraîne pas le modèle, on ne veut pas calculer les gradients.
    # Cela divise la consommation de RAM vidéo par 2 et accélère considérablement le calcul.
    with torch.no_grad():
        for images, labels in dataloader:
            # On envoie les données sur la carte graphique (ou le CPU)
            images = images.to(device)
            labels = labels.to(device)
            
            # Le modèle fait ses prédictions (sort des "logits")
            outputs = model(images)
            
            # On prend l'indice de la valeur la plus haute (0 = Swap, 1 = Reenactment)
            _, class_predite = torch.max(outputs, 1)
            
            # Mise à jour des compteurs
            total_images += labels.size(0)
            correct_predictions += (class_predite == labels).sum().item()
            
            # On stocke pour le rapport final
            toutes_les_predictions.extend(class_predite.cpu().numpy())
            tous_les_vrais_labels.extend(labels.cpu().numpy())

    # 3. Calcul de la précision globale
    accuracy = 100 * correct_predictions / total_images
    print(f"\n--- RÉSULTATS DE L'ÉVALUATION ---")
    print(f"Précision Globale (Accuracy) : {accuracy:.2f}% ({correct_predictions}/{total_images} images)")
    
    # 4. Affichage détaillé avec scikit-learn
    print("\nRapport Détaillé :")
    print(classification_report(
        tous_les_vrais_labels, 
        toutes_les_predictions, 
        target_names=["Face-Swap (0)", "Reenactment (1)"]
    ))
    
    return accuracy

In [24]:
# Assure-toi que ton modèle est sur le bon appareil (GPU ou CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
router_model = router_model.to(device)

# Lancement de l'évaluation
score_validation = evaluer_le_routeur(router_model, test_loader, device)

# RÈGLE D'OR N°3 : Si tu continues l'entraînement après ça, n'oublie pas de repasser en mode train !
# router_model.train()

Début de l'évaluation sur le set de validation...

--- RÉSULTATS DE L'ÉVALUATION ---
Précision Globale (Accuracy) : 83.16% (73890/88853 images)

Rapport Détaillé :
                 precision    recall  f1-score   support

  Face-Swap (0)       0.92      0.64      0.75     35776
Reenactment (1)       0.80      0.96      0.87     53077

       accuracy                           0.83     88853
      macro avg       0.86      0.80      0.81     88853
   weighted avg       0.85      0.83      0.82     88853



In [25]:
checkpoint_path = "router.pth"

checkpoint = {
    'epoch': 10,
    'model_state_dict': router_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item()
}

torch.save(checkpoint, checkpoint_path)
print("Checkpoint saved")

Checkpoint saved
